## 1. Clone Source Code UniVL từ GitHub

In [ ]:
# Clone UniVL repository
!git clone https://github.com/haohao0550/UniVL-Video-captioning.git
%cd UniVL-Video-captioning

## 2. Cài đặt dependencies

Cài đặt các thư viện cần thiết cho UniVL:

In [ ]:
!pip install tqdm boto3 requests pandas -q

In [ ]:
# ✅ Cài đặt pycocoevalcap để tính evaluation metrics (thay thế nlg-eval)
!pip install git+https://github.com/salaniz/pycocoevalcap.git -q

In [ ]:
# Thư viện cho Flan-T5 + LoRA
!pip install transformers peft accelerate -q

---
## 3. Chuẩn bị 

### 3.1 Download BERT base model files

In [ ]:
%%bash
# Tạo thư mục cho BERT
mkdir -p modules/bert-base-uncased
cd modules/bert-base-uncased

# Download vocab file
wget -q https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt
mv bert-base-uncased-vocab.txt vocab.txt

# Download BERT model
wget -q https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased.tar.gz
tar -xzf bert-base-uncased.tar.gz
rm bert-base-uncased.tar.gz

cd ../..
echo "✓ BERT model downloaded successfully!"
ls -lh modules/bert-base-uncased/

### 3.2 Tải UniVL pretrained weights

In [ ]:
%%bash
# Tạo thư mục weight
mkdir -p weight

# Download pretrained weight
echo "Downloading UniVL pretrained weight (this may take a few minutes)..."
wget -q https://github.com/microsoft/UniVL/releases/download/v0/univl.pretrained.bin -O weight/univl.pretrained.bin

echo "✓ UniVL pretrained weight downloaded!"
ls -lh weight/

### 3.3 Tải MSRVTT dataset

Dataset bao gồm:
- CSV files: Chứa thông tin về video IDs
- JSON file: Chứa captions và metadata  
- Pickle file: Chứa S3D video features (1024-dim, ~1.2GB)

In [ ]:
%%bash
# Tạo thư mục data
mkdir -p data

# Download MSRVTT dataset
echo "Downloading MSRVTT dataset (this will take several minutes, ~1.2GB)..."
wget -q https://github.com/microsoft/UniVL/releases/download/v0/msrvtt.zip -O data/msrvtt.zip

# Giải nén
echo "Extracting dataset..."
unzip -q data/msrvtt.zip -d data/

# Xóa file zip để tiết kiệm dung lượng
rm data/msrvtt.zip

echo "✓ MSRVTT dataset downloaded and extracted!"
ls -lh data/msrvtt/

---
## 4. Hiểu cấu trúc model UniVL

### 4.1 Kiến trúc model

In [ ]:
# Import các modules cần thiết
import sys
sys.path.append('./')

from modules.tokenization import BertTokenizer
from modules.modeling import UniVL
import torch

# Khởi tạo tokenizer
tokenizer = BertTokenizer.from_pretrained(
    "modules/bert-base-uncased", 
    do_lower_case=True
)

print(f"✓ Tokenizer vocab size: {len(tokenizer.vocab)}")
print(f"\nMẫu tokenization:")
sample_text = "A man is playing guitar on the stage"
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"  Text: {sample_text}")
print(f"  Tokens: {tokens}")
print(f"  Token IDs: {token_ids}")

---
## 5. Thiết lập training script

In [ ]:
from pathlib import Path

In [ ]:
output_dir = Path("./ckpts/ckpt_msrvtt_caption")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directory: {output_dir}")

### 4.2 Kiểm tra cấu hình GPU và distributed training

In [ ]:
import torch

n_gpu = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {n_gpu}")

if n_gpu > 0:
    for i in range(n_gpu):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
        
    # Khuyến nghị
    if n_gpu >= 4:
        print("\n✓ Có thể sử dụng 4 GPUs như trong README")
        print("  Lệnh: python -m torch.distributed.launch --nproc_per_node=4 main_task_caption.py ...")
    elif n_gpu > 1:
        print(f"\n⚠ Có {n_gpu} GPUs. Có thể điều chỉnh --nproc_per_node={n_gpu}")
    else:
        print("\n⚠ Chỉ có 1 GPU. Có thể train nhưng sẽ chậm hơn.")
        print("  Cân nhắc giảm batch_size nếu gặp OOM error")
else:
    print("\n✗ Không có GPU. CPU training sẽ rất chậm.")

---
## 6. Training Command

### 6.1 Command cho Multi-GPU Training (Khuyến nghị)

In [ ]:
import torch

# TRIỆT ĐỂ NCCL timeout trên Kaggle: luôn dùng single-process
n_gpu = torch.cuda.device_count()
nproc = 1

batch_size = 32 if n_gpu > 0 else 4
batch_size_val = 32 if n_gpu > 0 else 4
grad_accum = 2 if n_gpu > 0 else 1
lr = 1e-4

print(f"✓ n_gpu={n_gpu}, nproc={nproc}, batch_size={batch_size}, batch_size_val={batch_size_val}, grad_accum={grad_accum}, lr={lr}")

training_command = f"""torchrun --nproc_per_node={nproc} --standalone \\
main_task_caption.py \\
--do_train --do_eval --num_thread_reader=4 \\
--epochs=10 --batch_size={batch_size} \\
--batch_size_val={batch_size_val} \\
--gradient_accumulation_steps={grad_accum} \\
--lr {lr} --n_display=50 \\
--train_csv data/msrvtt/MSRVTT_train.9k.csv \\
--val_csv data/msrvtt/MSRVTT_JSFUSION_test.csv \\
--data_path data/msrvtt/MSRVTT_data.json \\
--features_path /kaggle/input/models/nguyhao/blip2-feature-extractor/other/default/1/msrvtt_blip2_features.pickle \\
--output_dir ckpts/ckpt_msrvtt_caption \\
--bert_model bert-base-uncased --do_lower_case \\
--datatype msrvtt --stage_two \\
--visual_num_hidden_layers 6 --cross_num_hidden_layers 2 \\
--llm_model google/flan-t5-base \\
--use_lora --lora_r 16 --lora_alpha 32 --lora_dropout 0.1 \\
--init_model /kaggle/input/models/nguyhao/blip2-ve-model/other/default/1/pytorch_model.bin.1 \\
--max_words 48 --max_frames 48 --video_dim 768
"""

print("\n" + "="*80)
print("TRAINING COMMAND (Kaggle GPU, Flan-T5 + LoRA):")
print("="*80)
print(training_command)
print("="*80)

with open("train_msrvtt_caption.sh", "w") as f:
    f.write(training_command)

print("\n✓ Đã lưu command vào train_msrvtt_caption.sh")

---
## 7. Chạy Training

### 7.1 Khởi tạo output directory

In [ ]:
# Tạo thư mục output
!mkdir -p ckpts/ckpt_msrvtt_caption
print("✓ Output directory created!")

### 7.2 Bắt đầu training

**Lưu ý quan trọng:**
- Training sẽ mất khoảng **3-6 giờ** tùy theo GPU
- Kaggle có giới hạn **9 giờ** GPU mỗi session
- Model sẽ được lưu checkpoint sau mỗi epoch
- Có thể dừng và resume nếu cần

In [ ]:
# %%time
# Chạy training script
!bash train_msrvtt_caption.sh